# Standalone Title and Summary Pipeline

This notebook is a separate pipeline and intentionally excludes taxonomy refresh logic.

Goal:
- Read markdown chat exports from `DeepSeek_Exports/`
- Generate `generated_title` first using local Ollama
- Generate `summary` second
- Save processed markdown files into `intermediate_markdowns/` without modifying originals

In [1]:
from __future__ import annotations

import json
import os
import re
import unicodedata
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import requests

## 1) Define Pipeline Scope and Contracts

This pipeline excludes taxonomy refresh and category generation.

Input contract per source markdown:
- `date` (frontmatter or filename prefix)
- `title` (original title)
- `conversation_id` (if present)
- `url` (if present)
- markdown body (chat history)

Output contract per processed markdown:
- `date`
- `original_title`
- `conversation_id`
- `url`
- `generated_title`
- `summary`
- `source_file`
- original body content

In [17]:
@dataclass
class PipelineConfig:
    project_root: Path
    export_dir: Path
    intermediate_dir: Path
    ollama_base_url: str
    ollama_model: str
    batch_size: int = 10
    max_input_chars: int = 12000
    retries: int = 2
    request_timeout_sec: int = 300


def load_config() -> PipelineConfig:
    root = Path.cwd()
    cfg = PipelineConfig(
        project_root=root,
        export_dir=root / "DeepSeek_Exports",
        intermediate_dir=root / "intermediate_markdowns",
        ollama_base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
        ollama_model=os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct"),
        batch_size=int(os.getenv("PIPELINE_BATCH_SIZE", "10")),
        max_input_chars=int(os.getenv("PIPELINE_MAX_INPUT_CHARS", "12000")),
        retries=int(os.getenv("PIPELINE_RETRIES", "2")),
        request_timeout_sec=int(os.getenv("PIPELINE_REQUEST_TIMEOUT_SEC", "300")),
    )

    if not cfg.export_dir.exists():
        raise FileNotFoundError(f"Missing export directory: {cfg.export_dir}")

    cfg.intermediate_dir.mkdir(exist_ok=True)
    return cfg


config = load_config()
print({
    "export_dir": str(config.export_dir),
    "intermediate_dir": str(config.intermediate_dir),
    "ollama_model": config.ollama_model,
    "batch_size": config.batch_size,
    "request_timeout_sec": config.request_timeout_sec,
})

{'export_dir': 'c:\\Users\\HELEN1822\\OneDrive - Willis Towers Watson\\Documents\\Github\\MindForge\\DeepSeek_Exports', 'intermediate_dir': 'c:\\Users\\HELEN1822\\OneDrive - Willis Towers Watson\\Documents\\Github\\MindForge\\intermediate_markdowns', 'ollama_model': 'qwen2.5:7b-instruct', 'batch_size': 10, 'request_timeout_sec': 300}


## 2) Set Up Configuration and Environment Variables

Configuration is loaded from environment variables with safe defaults.

Required runtime dependency:
- Local Ollama running at `OLLAMA_BASE_URL`
- Model available in Ollama tags (default: `qwen2.5:7b-instruct`)

In [18]:
def preflight_ollama(cfg: PipelineConfig, raise_on_fail: bool = True) -> dict[str, Any]:
    result = {"ok": True, "message": ""}
    try:
        r = requests.get(f"{cfg.ollama_base_url}/api/tags", timeout=8)
        r.raise_for_status()
        models = [m.get("name", "") for m in r.json().get("models", [])]
        found = any(name == cfg.ollama_model or name.startswith(f"{cfg.ollama_model}:") for name in models)
        if not found:
            result["ok"] = False
            result["message"] = f"Model not found: {cfg.ollama_model}. Available: {models[:8]}"
        else:
            result["message"] = f"Ollama ready with model {cfg.ollama_model}"
    except requests.RequestException as e:
        result["ok"] = False
        result["message"] = f"Cannot reach Ollama at {cfg.ollama_base_url}. Error: {e}"

    if raise_on_fail and not result["ok"]:
        raise RuntimeError(result["message"])
    return result


def _parse_json_robust(text: str) -> dict[str, Any]:
    cleaned = text.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    candidates = [cleaned]

    m = re.search(r"\{[\s\S]*\}", cleaned)
    if m:
        candidates.append(m.group(0))

    for cand in list(candidates):
        candidates.append(re.sub(r",\s*([}\]])", r"\1", cand))

    for cand in candidates:
        try:
            out = json.loads(cand)
            if isinstance(out, dict):
                return out
        except json.JSONDecodeError:
            continue

    preview = cleaned[:400]
    raise RuntimeError(f"Model returned non-parseable JSON. Preview: {preview}")


def ollama_call_json(cfg: PipelineConfig, system_prompt: str, user_prompt: str, temperature: float = 0.2) -> dict[str, Any]:
    payload = {
        "model": cfg.ollama_model,
        "prompt": f"<|system|>\n{system_prompt}\n<|user|>\n{user_prompt}\n<|assistant|>",
        "stream": False,
        "options": {"temperature": temperature},
        "format": "json",
    }

    last_err: Exception | None = None
    for attempt in range(cfg.retries + 1):
        try:
            r = requests.post(
                f"{cfg.ollama_base_url}/api/generate",
                json=payload,
                timeout=cfg.request_timeout_sec,
            )
            r.raise_for_status()
            text = r.json().get("response", "{}")
            return _parse_json_robust(text)
        except (requests.RequestException, RuntimeError) as e:
            last_err = e
            if attempt >= cfg.retries:
                break

    raise RuntimeError(f"Ollama call failed after retries: {last_err}")

## 3) Implement Data Ingestion Stage

Load all markdown files, parse frontmatter/body, and normalize into in-memory records.

In [4]:
def parse_date_from_filename(filename: str) -> str:
    m = re.match(r"^(\d{4}-\d{2}-\d{2})", filename)
    return m.group(1) if m else ""


def extract_frontmatter_and_body(text: str) -> tuple[dict[str, Any], str]:
    if text.startswith("---\n"):
        parts = text.split("---\n", 2)
        if len(parts) == 3:
            fm_raw, body = parts[1], parts[2]
            fm = {}
            for line in fm_raw.splitlines():
                if ":" in line:
                    k, v = line.split(":", 1)
                    fm[k.strip()] = v.strip().strip('"')
            return fm, body
    return {}, text


def ingest_exports(cfg: PipelineConfig) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for path in sorted(cfg.export_dir.glob("*.md")):
        raw = path.read_text(encoding="utf-8", errors="ignore")
        fm, body = extract_frontmatter_and_body(raw)
        rows.append({
            "source_file": path.name,
            "date": fm.get("date", "") or parse_date_from_filename(path.name),
            "original_title": fm.get("title", path.stem),
            "conversation_id": fm.get("conversation_id", ""),
            "url": fm.get("url", ""),
            "body": body,
        })
    return rows


records = ingest_exports(config)
print(f"Loaded {len(records)} markdown files")
if records:
    print({k: records[0][k] for k in ["source_file", "date", "original_title", "conversation_id", "url"]})

Loaded 40 markdown files
{'source_file': '2026-03-08 制定个性化护肤方案需求清单.md', 'date': '2026-03-08', 'original_title': '2026-03-08 制定个性化护肤方案需求清单', 'conversation_id': '45d17a7a-bd02-46de-b845-e4669ff86712', 'url': 'https://chat.deepseek.com/a/45d17a7a-bd02-46de-b845-e4669ff86712'}


## 4) Implement Transformation Stage (Excluding Taxonomy Refresh)

Transformations in this stage only:
- Generate `generated_title` first
- Generate `summary` second
- Deduplicate by `(conversation_id, original_title, body)` key

No taxonomy/category logic is used.

In [19]:
def slugify_filename(text: str, max_len: int = 80) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\\/:*?\"<>|]", " ", text)
    text = re.sub(r"\s+", "-", text).strip("-._ ")
    text = text[:max_len].strip("-._ ")
    return text or "untitled"


def generate_title(cfg: PipelineConfig, body: str, original_title: str) -> str:
    system_prompt = "Generate concise best-fit title for this chat history. Return JSON only."
    user_prompt = json.dumps({
        "task": "generate_title",
        "original_title": original_title,
        "chat_history": body[: cfg.max_input_chars],
        "rules": [
            "max 14 words",
            "no quotes",
            "avoid generic words like Chat/Notes",
            "avoid date-only title",
            "use dominant language of content",
        ],
        "output_schema": {"generated_title": "string"},
    }, ensure_ascii=False)

    out = ollama_call_json(cfg, system_prompt, user_prompt, temperature=0.2)
    title = str(out.get("generated_title", "")).strip()
    return (title or original_title or "Untitled Conversation")[:120]


def generate_summary(cfg: PipelineConfig, body: str, generated_title: str) -> str:
    system_prompt = "Summarize this chat history in one short paragraph. Return JSON only."
    user_prompt = json.dumps({
        "task": "generate_summary",
        "generated_title": generated_title,
        "chat_history": body[: cfg.max_input_chars],
        "rules": ["2-4 sentences", "no bullet points", "preserve asks and outcomes"],
        "output_schema": {"summary": "string"},
    }, ensure_ascii=False)

    out = ollama_call_json(cfg, system_prompt, user_prompt, temperature=0.2)
    summary = str(out.get("summary", "")).strip()
    return (summary or "Summary unavailable.")[:1200]


def transform_records(cfg: PipelineConfig, rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    seen: set[tuple[str, str, str]] = set()
    output: list[dict[str, Any]] = []

    for idx, row in enumerate(rows, start=1):
        key = (str(row.get("conversation_id", "")), str(row.get("original_title", "")), str(row.get("body", "")))
        if key in seen:
            continue
        seen.add(key)

        try:
            generated_title = generate_title(cfg, row["body"], row["original_title"])
            summary = generate_summary(cfg, row["body"], generated_title)
            error = ""
        except Exception as e:
            generated_title = row["original_title"] or "Untitled Conversation"
            summary = "Summary unavailable due to model timeout or parse error."
            error = str(e)[:300]

        output.append({
            **row,
            "generated_title": generated_title,
            "summary": summary,
            "error": error,
        })

        if idx % 5 == 0:
            print(f"[transform-progress] {idx}/{len(rows)}")

    return output

## 5) Implement Validation and Quality Checks

Checks:
- required fields exist
- generated title and summary are non-empty
- output record uniqueness by `source_file`

In [6]:
def validate_records(rows: list[dict[str, Any]]) -> None:
    if not rows:
        raise ValueError("No transformed rows to validate")

    required = {"source_file", "original_title", "generated_title", "summary", "body"}
    source_seen: set[str] = set()

    for i, row in enumerate(rows, start=1):
        missing = [k for k in required if not row.get(k)]
        if missing:
            raise ValueError(f"Row {i} missing required fields: {missing}")

        src = str(row["source_file"])
        if src in source_seen:
            raise ValueError(f"Duplicate source_file found after dedup: {src}")
        source_seen.add(src)

## 6) Implement Output/Publish Stage

Write each processed record to `intermediate_markdowns/` with idempotent naming.

## 7) Orchestrate the Separate Pipeline Flow

Compose all stages with logging and a single callable entrypoint.

## 8) Add Unit Tests for Each Pipeline Stage

Use lightweight assertions on fixtures.

## 9) Run the Pipeline and Capture Execution Logs

Run preflight and then execute the pipeline end-to-end.

In [21]:
def yaml_escape(value: str) -> str:
    value = str(value).replace("\\", "\\\\").replace('"', '\\"')
    return f'"{value}"'


def build_frontmatter(row: dict[str, Any]) -> str:
    ordered = ["date", "original_title", "conversation_id", "url", "generated_title", "summary", "source_file"]
    lines = ["---"]
    for key in ordered:
        lines.append(f"{key}: {yaml_escape(row.get(key, ''))}")
    lines.append("---")
    return "\n".join(lines)


def _title_for_filename(row: dict[str, Any]) -> str:
    title = str(row.get("generated_title", "")).strip()
    title = re.sub(r"^\d{4}-\d{2}-\d{2}\s*", "", title)
    title = title.strip("- _")
    return title or str(row.get("original_title", "")).strip() or "untitled"


def publish_rows(cfg: PipelineConfig, rows: list[dict[str, Any]], overwrite: bool = False) -> list[dict[str, str]]:
    outputs: list[dict[str, str]] = []

    for row in rows:
        prefix = f"{row.get('date', '')} " if row.get("date") else ""
        filename = f"{prefix}{slugify_filename(_title_for_filename(row))}.md"
        path = cfg.intermediate_dir / filename

        if path.exists() and not overwrite:
            stem, suffix, i = path.stem, path.suffix, 2
            while True:
                cand = cfg.intermediate_dir / f"{stem}_{i}{suffix}"
                if not cand.exists():
                    path = cand
                    break
                i += 1

        content = build_frontmatter(row) + "\n\n" + row["body"].lstrip("\n")
        path.write_text(content, encoding="utf-8")
        outputs.append({"source_file": row["source_file"], "output_file": str(path.name)})

    return outputs


def _append_jsonl(path: Path, payload: dict[str, Any]) -> None:
    line = json.dumps(payload, ensure_ascii=False)
    with path.open("a", encoding="utf-8") as f:
        f.write(line + "\n")


def _read_completed_sources(checkpoint_path: Path) -> set[str]:
    completed: set[str] = set()
    if not checkpoint_path.exists():
        return completed

    for line in checkpoint_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        if not line.strip():
            continue
        try:
            row = json.loads(line)
            if row.get("status") == "ok":
                completed.add(str(row.get("source_file", "")))
        except json.JSONDecodeError:
            continue
    return completed


def run_pipeline(
    cfg: PipelineConfig,
    limit: int | None = None,
    overwrite: bool = False,
    checkpoint_file: str = "_pipeline_results.jsonl",
    resume_from_checkpoint: bool = True,
) -> dict[str, Any]:
    started = datetime.now(timezone.utc)
    print(f"[start] {started.isoformat()}")

    pre = preflight_ollama(cfg, raise_on_fail=False)
    print("[preflight]", pre)
    if not pre["ok"]:
        raise RuntimeError(pre["message"])

    checkpoint_path = cfg.intermediate_dir / checkpoint_file
    completed_sources = _read_completed_sources(checkpoint_path) if resume_from_checkpoint else set()

    rows = ingest_exports(cfg)
    if limit is not None:
        rows = rows[:limit]

    if completed_sources:
        rows = [r for r in rows if r.get("source_file") not in completed_sources]
        print(f"[resume] skipped={len(completed_sources)} already completed from checkpoint")

    print(f"[ingest] rows={len(rows)}")

    output_count = 0
    failure_count = 0
    samples: list[dict[str, str]] = []

    for idx, row in enumerate(rows, start=1):
        try:
            transformed = transform_records(cfg, [row])
            validate_records(transformed)
            published = publish_rows(cfg, transformed, overwrite=overwrite)
            out = published[0]
            output_count += 1
            if len(samples) < 5:
                samples.append(out)

            _append_jsonl(checkpoint_path, {
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "status": "ok",
                "source_file": row.get("source_file", ""),
                "output_file": out.get("output_file", ""),
                "generated_title": transformed[0].get("generated_title", ""),
                "summary": transformed[0].get("summary", ""),
                "error": transformed[0].get("error", ""),
            })
        except Exception as e:
            failure_count += 1
            _append_jsonl(checkpoint_path, {
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "status": "failed",
                "source_file": row.get("source_file", ""),
                "output_file": "",
                "generated_title": "",
                "summary": "",
                "error": str(e)[:500],
            })

        if idx % 5 == 0 or idx == len(rows):
            print(f"[run-progress] {idx}/{len(rows)}")

    ended = datetime.now(timezone.utc)
    print(f"[publish] files={output_count}, failed={failure_count}")

    return {
        "started_at": started.isoformat(),
        "ended_at": ended.isoformat(),
        "input_count": len(rows),
        "output_count": output_count,
        "failure_count": failure_count,
        "checkpoint_file": str(checkpoint_path),
        "sample": samples,
    }

In [8]:
# --- Local unit checks (no model call) ---
fixture = {
    "source_file": "2026-01-01 Demo.md",
    "date": "2026-01-01",
    "original_title": "Old Title",
    "conversation_id": "abc",
    "url": "https://example.com",
    "generated_title": "New Better Title",
    "summary": "A short summary.",
    "body": "Chat body",
}

assert parse_date_from_filename("2026-01-01 demo.md") == "2026-01-01"
assert slugify_filename("Hello World!") == "Hello-World"
assert "generated_title" in build_frontmatter(fixture)
print("Local tests passed")

Local tests passed


In [22]:
# --- One-file quality validation run ---
one_result = run_pipeline(config, limit=1, overwrite=False)
print(json.dumps(one_result, ensure_ascii=False, indent=2))

if not one_result.get("sample"):
    raise RuntimeError("No output file generated in one-file run.")

out_file = config.intermediate_dir / one_result["sample"][0]["output_file"]
out_text = out_file.read_text(encoding="utf-8", errors="ignore")
print("\nOutput file:", out_file)
print("\nPreview:\n")
print(out_text[:1200])

required_keys = [
    "date:",
    "original_title:",
    "conversation_id:",
    "url:",
    "generated_title:",
    "summary:",
    "source_file:",
]
for key in required_keys:
    assert key in out_text, f"Missing key in output frontmatter: {key}"

print("\nQuality validation passed for one file")

[start] 2026-06-25T15:03:15.371129+00:00
[preflight] {'ok': True, 'message': 'Ollama ready with model qwen2.5:7b-instruct'}
[ingest] rows=1
[run-progress] 1/1
[publish] files=1, failed=0
{
  "started_at": "2026-06-25T15:03:15.371129+00:00",
  "ended_at": "2026-06-25T15:03:55.936448+00:00",
  "input_count": 1,
  "output_count": 1,
  "failure_count": 0,
  "checkpoint_file": "c:\\Users\\HELEN1822\\OneDrive - Willis Towers Watson\\Documents\\Github\\MindForge\\intermediate_markdowns\\_pipeline_results.jsonl",
  "sample": [
    {
      "source_file": "2026-03-08 制定个性化护肤方案需求清单.md",
      "output_file": "2026-03-08 偏干性肤质护理方案_2.md"
    }
  ]
}

Output file: c:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\intermediate_markdowns\2026-03-08 偏干性肤质护理方案_2.md

Preview:

---
date: "2026-03-08"
original_title: "2026-03-08 制定个性化护肤方案需求清单"
conversation_id: "45d17a7a-bd02-46de-b845-e4669ff86712"
url: "https://chat.deepseek.com/a/45d17a7a-bd02-46de-b845-e4669ff86712"
generated_

In [20]:
# --- Full batch run ---
full_result = run_pipeline(config, limit=None, overwrite=False)
print(json.dumps(full_result, ensure_ascii=False, indent=2))

[start] 2026-06-25T14:50:27.791440+00:00
[preflight] {'ok': True, 'message': 'Ollama ready with model qwen2.5:7b-instruct'}
[ingest] rows=40
[transform-progress] 5/40
[transform-progress] 10/40
[transform-progress] 15/40
[transform-progress] 20/40
[transform-progress] 25/40
[transform-progress] 30/40
[transform-progress] 35/40
[transform-progress] 40/40
[transform] rows=40
[validate] ok
[publish] files=40
{
  "started_at": "2026-06-25T14:50:27.791440+00:00",
  "ended_at": "2026-06-25T15:01:39.328256+00:00",
  "input_count": 40,
  "output_count": 40,
  "sample": [
    {
      "source_file": "2026-03-08 制定个性化护肤方案需求清单.md",
      "output_file": "2026-03-08 偏干性肤质护理方案.md"
    },
    {
      "source_file": "2026-03-25 Hair Consultation Questions.md",
      "output_file": "2026-03-25 巴黎画染+侧分长刘海.md"
    },
    {
      "source_file": "2026-03-25 马尔代夫旅游打包清单.md",
      "output_file": "2026-03-25 马尔代夫旅游详尽打包清单.md"
    },
    {
      "source_file": "2026-03-26 Bathroom Design Prompt Assistance.md",
 